# Obtencion de datos mediante API Publica (*nba_api*) y configurar estructura de datos para creacion de objetos (*nba_api.stats.endpoints*)

Vamos a realizar un proceso ***ETL(Extract, Transform, Load)*** para generar un archivo csv, con todos los datos que necesitamos.

Haremos un metodo con una serie de pasos para **testear codigo escalable y generar un archivo unico final con la informacion necesaria**

El orden de los pasos para el desarrollo del archivo sera:

- 1º conectar a una API publica donde poder obtener datos
- 2º Realizar una prueba de que la conexion es estable al realizar una consulta
- 3º Realizar una prueba para capturar datos parciales en base a una muestra y convertirlos a .csv (**datos de los atributos dinamicos**)
- 4º Completar la captura de los totales de la muestra y completar el .csv (**datos de los atributos fijos**)
- 5º Completar la captura con todos los jugadores y obtener un .csv definitivo.

## 1º Conexion nba_api

Una vez definido la clase Jugador que compondran los objetos que posteriormente nos permitira realizar analisis y obtener resultados.

Los datos vamos a solicitarlos traer para exportarlos posteriormente en archivos .csv, que leeremos con pandas.

En primer lugar instalamos una libreria que nos permitira entablar una conexion mediante APIs a una BBDD publica de datos de la NBA

**En este caso sera nba_api**

In [6]:
!pip install nba_api

## 2º Prueba de indexacion de datos mediante un ID

Realizamos una prueba de la libreria y obtenemos datos para ver que la conexion funciona

> Aqui utilizamos 2 metodos propios como son *get_players* que nos devuelve un listado de jugadores y *PlayerGameLog* que nos devuelve la informacion de estadisticas de un jugador en cada partido

In [7]:
from nba_api.stats.static import players
from nba_api.stats.endpoints import playergamelog
import pandas as pd

# 1. Buscar el ID de LeBron James (Necesitamos su ID numérico)
player_dict = players.get_players()
lebron = [player for player in player_dict if player['full_name'] == 'LeBron James'][0]
lebron_id = lebron['id']

print(f"ID de LeBron encontrado: {lebron_id}")

# 2. Pedir sus estadísticas de la temporada actual (2024-25)
# El endpoint 'playergamelog' nos da el 'box score' de cada partido
gamelog = playergamelog.PlayerGameLog(player_id=lebron_id, season='2024-25')

# 3. Convertir a DataFrame
df_lebron = gamelog.get_data_frames()[0]

# 4. Mostrar las columnas para ver si tenemos lo que necesitamos
print("\n--- Columnas disponibles ---")
print(df_lebron.columns.tolist())

# 5. Mostrar datos del último partido
print("\n--- Último partido ---")
print(df_lebron[['GAME_DATE', 'MATCHUP', 'PTS', 'AST', 'REB', 'STL', 'BLK', 'TOV']].head(1))

ID de LeBron encontrado: 2544

--- Columnas disponibles ---
['SEASON_ID', 'Player_ID', 'Game_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE']

--- Último partido ---
      GAME_DATE      MATCHUP  PTS  AST  REB  STL  BLK  TOV
0  Apr 11, 2025  LAL vs. HOU   14    8    4    1    0    1


## 3º Realizamos una captura de prueba de datos para pasarlos a csv.

Ahora vamos a establecer un bloque de codigo que permita capturar datos y exportarlos a un archivo .csv, parcialmente, **unicamente del bloque de estaditicas que compondran los objetos que vamos a generar** (PTS, AST,...)

> Dado que esta API tiene un limite de peticiones, añadiremos un *delay* para que no nos bloquee el acceso a los datos.
>
> Vamos a seleccionar una muestra de 5 jugadores, para realizar una prueba de que funciona y optimizar el tiempo de carga del programa

In [1]:
import pandas as pd
import time
from nba_api.stats.static import players
from nba_api.stats.endpoints import playergamelog

# 1. Definimos una lista de IDs de jugadores clave para el Portfolio
# (Hacemos esto manual ahora para probar rápido. Luego podemos automatizar todos).
# IDs: LeBron (2544), Curry (201939), Doncic (1629029), Jokic (203999), Giannis (203507)
target_ids = [2544, 201939, 1629029, 203999, 203507]

print(f"Iniciando descarga para {len(target_ids)} jugadores...")

# Lista vacía para ir guardando los dataframes de cada jugador
lista_dfs = []

# 2. Bucle FOR para recorrer cada jugador 
for pid in target_ids:
    try:
        print(f"Descargando datos del ID: {pid}...", end=" ")
        
        # Llamada a la API
        gamelog = playergamelog.PlayerGameLog(player_id=pid, season='2024-25')
        df_temp = gamelog.get_data_frames()[0]
        
        # Añadimos columna con el ID para no perder quién es quién
        df_temp['Player_ID'] = pid
        
        # Guardamos en la lista
        lista_dfs.append(df_temp)
        print("OK.")
        
        # PAUSA DE CORTESÍA (600ms) para evitar bloqueo de API
        time.sleep(0.6) 
        
    except Exception as e:
        print(f"Error con {pid}: {e}")

# 3. Concatenar (Unir) todos los datos en una sola Tabla Maestra 
if lista_dfs:
    df_total = pd.concat(lista_dfs, ignore_index=True)
    
    # 4. Guardar a CSV (Persistencia) 
    nombre_archivo = '5_samples_nba_raw_data.csv'
    df_total.to_csv(nombre_archivo, index=False)
    
    print(f"\n¡ÉXITO! Se han guardado {len(df_total)} filas en '{nombre_archivo}'.")
    print("Muestra de datos:")
    print(df_total[['Player_ID', 'GAME_DATE', 'PTS', 'AST', 'REB']].head())
else:
    print("\nNo se descargaron datos.")

Iniciando descarga para 5 jugadores...
OK.cargando datos del ID: 2544... 
Descargando datos del ID: 201939... OK.
Descargando datos del ID: 1629029... OK.
Descargando datos del ID: 203999... OK.
Descargando datos del ID: 203507... OK.

¡ÉXITO! Se han guardado 327 filas en '5_samples_nba_raw_data.csv'.
Muestra de datos:
   Player_ID     GAME_DATE  PTS  AST  REB
0       2544  Apr 11, 2025   14    8    4
1       2544  Apr 09, 2025   27    3    7
2       2544  Apr 08, 2025   28    3    7
3       2544  Apr 06, 2025   19    7    3
4       2544  Apr 04, 2025   27    8    0


## 4º Obtencion completa de informacion de los jugadores del muestreo

Tras ver que resultado funciona y podemos obtener datos estadisticos, ahora vamos a obtener otros datos **los relacionados con la identificacion del jugador** (nacionalidad, edad, altura,...)

> Volvemos a realizar el muestreo con 5 jugadores para mejorar la facilidad de carga del programa
>
> Primero capturaremos de nuevo los datos de estadisticas (**futuros atributos dinamicos de los jugadores**)en un dataframe identificado por jugador, luego capturaremos datos identificativos (**futuso atributos fijos**) en otro dataframe identificado por jugador
>
> Segundo, aunaremos los 2 dataframes en un unico dataframe.
>
> Tercero, descargaremos un archivo csv conla informacion

In [2]:
import pandas as pd
import time
from nba_api.stats.endpoints import playergamelog, commonplayerinfo

# --- CONFIGURACIÓN ---
# IDs de muestra (LeBron, Curry, Doncic, Jokic, Giannis)
target_ids = [2544, 201939, 1629029, 203999, 203507]
lista_stats = []
lista_bios = []

print(f"Iniciando proceso ETL para {len(target_ids)} jugadores...")

# --- FASE 1: EXTRACCIÓN (Extract) ---
for pid in target_ids:
    print(f"Procesando ID: {pid}...", end=" ")
    
    try:
        # A. Obtener Stats (Dinámicas)
        gamelog = playergamelog.PlayerGameLog(player_id=pid, season='2024-25')
        df_stats = gamelog.get_data_frames()[0]
        
        # Agrupamos YA las stats para tener 1 fila por jugador
        # (Aquí simplificamos el proceso de agrupación que hicimos antes)
        cols_numericas = ['PTS', 'AST', 'REB', 'STL', 'BLK', 'TOV', 'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA']
        # Filtramos solo columnas que existen para evitar errores
        cols_validas = [c for c in cols_numericas if c in df_stats.columns]
        
        # Serie con los totales sumados
        stats_agrupadas = df_stats[cols_validas].sum()
        # Convertimos a DataFrame de 1 fila y añadimos ID
        df_resumen = stats_agrupadas.to_frame().T
        df_resumen['PLAYER_ID'] = pid # Llave para cruzar
        
        lista_stats.append(df_resumen)

        # B. Obtener Biografía (Fijas)
        # Pausa para no saturar API entre llamadas del mismo jugador
        time.sleep(0.4) 
        bio_info = commonplayerinfo.CommonPlayerInfo(player_id=pid)
        df_bio = bio_info.get_data_frames()[0]
        
        # Seleccionamos solo lo que nos interesa
        cols_bio = ['PERSON_ID', 'DISPLAY_FIRST_LAST', 'COUNTRY', 'HEIGHT', 'POSITION', 'TEAM_ABBREVIATION', 'BIRTHDATE']
        df_bio_limpio = df_bio[cols_bio]
        # Renombramos PERSON_ID a PLAYER_ID para que coincida con la otra tabla
        df_bio_limpio = df_bio_limpio.rename(columns={'PERSON_ID': 'PLAYER_ID'})
        
        lista_bios.append(df_bio_limpio)
        
        print("OK.")
        time.sleep(0.4) # Pausa entre jugadores

    except Exception as e:
        print(f"Error con {pid}: {e}")

# --- FASE 2: TRANSFORMACIÓN (Transform) ---
if lista_stats and lista_bios:
    # Concatenar todos los trozos en dos grandes tablas
    df_tabla_stats = pd.concat(lista_stats, ignore_index=True)
    df_tabla_bios = pd.concat(lista_bios, ignore_index=True)
    
    print("\nRealizando cruce de datos (Merge)...")
    # MERGE: Es como un BUSCARV o JOIN. Unimos las tablas usando 'PLAYER_ID' como nexo.
    # Combinando datos: merge y join
    df_final = pd.merge(df_tabla_bios, df_tabla_stats, on='PLAYER_ID', how='inner')
    
    # Cálculos finales (Ingeniería de características) antes de guardar
    df_final['Edad'] = 2026 - df_final['BIRTHDATE'].str[:4].astype(int)
    
    # --- FASE 3: CARGA (Load) ---
    nombre_archivo = '5_samples_nba_final_dataset.csv'
    df_final.to_csv(nombre_archivo, index=False)
    
    print(f"¡ÉXITO! Archivo maestro generado: {nombre_archivo}")
    print("Columnas:", df_final.columns.tolist())
    display(df_final.head())
else:
    print("No se pudieron obtener datos.")

Iniciando proceso ETL para 5 jugadores...
OK.cesando ID: 2544... 
Procesando ID: 201939... OK.
Procesando ID: 1629029... OK.
Procesando ID: 203999... OK.
Procesando ID: 203507... OK.

Realizando cruce de datos (Merge)...
¡ÉXITO! Archivo maestro generado: 5_samples_nba_final_dataset.csv
Columnas: ['PLAYER_ID', 'DISPLAY_FIRST_LAST', 'COUNTRY', 'HEIGHT', 'POSITION', 'TEAM_ABBREVIATION', 'BIRTHDATE', 'PTS', 'AST', 'REB', 'STL', 'BLK', 'TOV', 'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA', 'Edad']


,PLAYER_ID,DISPLAY_FIRST_LAST,COUNTRY,HEIGHT,POSITION,TEAM_ABBREVIATION,BIRTHDATE,PTS,AST,REB,STL,BLK,TOV,FGM,FGA,FG3M,FG3A,FTM,FTA,Edad
0,2544,LeBron James,USA,6-9,Forward,LAL,1984-12-30T00:00:00,1710,575,546,70,39,260,651,1270,149,396,259,331,42
1,201939,Stephen Curry,USA,6-2,Guard,GSW,1988-03-14T00:00:00,1718,421,310,80,30,200,564,1258,311,784,279,299,38
2,1629029,Luka Dončić,Slovenia,6-8,Forward-Guard,LAL,1999-02-28T00:00:00,1408,383,409,89,21,179,461,1025,177,481,309,395,27
3,203999,Nikola Jokić,Serbia,6-11,Center,DEN,1995-02-19T00:00:00,2071,716,892,127,45,230,786,1364,138,331,361,451,31
4,203507,Giannis Antetokounmpo,Greece,6-11,Forward,MIL,1994-12-06T00:00:00,2036,433,798,58,78,206,793,1319,14,63,436,707,32


## 5º Realizamos una solicitud de todos los datos para todos los jugadores y generamos un archivo final .csv

Finalmente generamos un codigo que permita realizar un archivo .csv con la informacion de los +500 jugadores de la NBA con la estructura de datos que hemos definido anteriormente.

> Vamos a incluir un contador de progreso ya que es un progreso que llevara su tiempo de ejecucion
>
> Vamos a realizarlo de la temporada 2025-2026 ya que es la que nos interesa para el ejercicio.

In [ ]:
import pandas as pd
import time
from nba_api.stats.static import players
from nba_api.stats.endpoints import playergamelog, commonplayerinfo

# --- CONFIGURACIÓN ---
TEMPORADA = '2025-26'  # La temporada que interesa obtener los datos
lista_stats = []
lista_bios = []

print(f"--- INICIANDO PROCESO MASIVO NBA ({TEMPORADA}) ---")

# 1. OBTENER LISTA DE TODOS LOS JUGADORES ACTIVOS
# En lugar de poner IDs a mano, pedimos el censo completo a la API.
print("Descargando lista de jugadores activos...", end=" ")
todos_jugadores = players.get_active_players()
print(f"¡Hecho! Se han encontrado {len(todos_jugadores)} jugadores en plantilla.")

# --- FASE 1: EXTRACCIÓN (Bucle Masivo) ---
# Usamos enumerate para tener un contador (i) y mostrar el progreso
for i, jugador in enumerate(todos_jugadores):
    pid = jugador['id']
    nombre = jugador['full_name']
    
    # Imprimimos progreso cada 10 jugadores para no ensuciar la pantalla, 
    # o si hay error.
    if i % 10 == 0:
        print(f"Procesando {i}/{len(todos_jugadores)}: {nombre} ({pid})...")
    
    try:
        # A. Obtener Stats (GameLog)
        gamelog = playergamelog.PlayerGameLog(player_id=pid, season=TEMPORADA)
        df_stats = gamelog.get_data_frames()[0]
        
        # SI EL JUGADOR NO HA JUGADO (Ej: Rookies en banquillo o lesionados todo el año)
        # El DataFrame estará vacío. Lo saltamos.
        if df_stats.empty:
            continue

        # Agrupamos stats (Transformación rápida)
        cols_numericas = ['PTS', 'AST', 'REB', 'STL', 'BLK', 'TOV', 'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA']
        cols_validas = [c for c in cols_numericas if c in df_stats.columns]
        
        stats_agrupadas = df_stats[cols_validas].sum()
        df_resumen = stats_agrupadas.to_frame().T
        df_resumen['PLAYER_ID'] = pid 
        
        lista_stats.append(df_resumen)

        # B. Obtener Biografía (CommonPlayerInfo)
        # Pequeña pausa para evitar bloqueo de API (Rate Limit)
        time.sleep(0.6) 
        
        bio_info = commonplayerinfo.CommonPlayerInfo(player_id=pid)
        df_bio = bio_info.get_data_frames()[0]
        
        # Limpieza de columnas bio
        cols_bio = ['PERSON_ID', 'DISPLAY_FIRST_LAST', 'COUNTRY', 'HEIGHT', 'POSITION', 'TEAM_ABBREVIATION', 'BIRTHDATE']
        # Aseguramos que existan antes de filtrar
        cols_bio_validas = [c for c in cols_bio if c in df_bio.columns]
        df_bio_limpio = df_bio[cols_bio_validas]
        
        df_bio_limpio = df_bio_limpio.rename(columns={'PERSON_ID': 'PLAYER_ID'})
        lista_bios.append(df_bio_limpio)
        
        # Pausa entre jugadores
        time.sleep(0.6)

    except Exception as e:
        # Si falla uno, no paramos el programa. Lo registramos y seguimos.
        print(f"Error con {nombre}: {e}")

# --- FASE 2 y 3: TRANSFORMACIÓN Y CARGA ---
print("\n--- FINALIZANDO PROCESO ---")

if lista_stats and lista_bios:
    print("Concatenando tablas...")
    df_tabla_stats = pd.concat(lista_stats, ignore_index=True)
    df_tabla_bios = pd.concat(lista_bios, ignore_index=True)
    
    print("Cruzando datos (Merge)...")
    # Combinando datos: merge
    df_final = pd.merge(df_tabla_bios, df_tabla_stats, on='PLAYER_ID', how='inner')
    
    # Cálculo de Edad
    # Convertimos el año de nacimiento (String) a entero y restamos al año actual
    if 'BIRTHDATE' in df_final.columns:
        df_final['Edad'] = 2026 - df_final['BIRTHDATE'].str[:4].astype(float)
    
    # Guardar a CSV
    nombre_archivo = 'complete_nba_final_dataset_2026.csv'
    df_final.to_csv(nombre_archivo, index=False) 
    
    print(f"¡ÉXITO TOTAL! Base de datos generada: {nombre_archivo}")
    print(f"Jugadores procesados correctamente: {len(df_final)}")
    display(df_final.head())
    
else:
    print("No se pudieron obtener datos. Revisa tu conexión o la API.")

**La API nos ha dado error al intentar realizar tantas solicitudes** pese a nuestros esfuerzos de evitar que nos echara añadiendo un delay.

> Vamos a re-estructurar el codigo para que se vaya guardando 1  a 1 cada registro de jugador tal y como necesitamos
>
> Ademas tendremos una confirmacion de los jugadores que ya estan descargados en el archivo
>
> La idea es que cuando salte la API, al reiniciar el programa, compruebe cual fue el ultimo registro que genero y continue a partir de este.

In [1]:
import pandas as pd
import time
import os
from nba_api.stats.static import players
from nba_api.stats.endpoints import playergamelog, commonplayerinfo, leaguestandings

# --- CONFIGURACIÓN ---
TEMPORADA = '2025-26'
NOMBRE_ARCHIVO = 'nba_final_dataset_2026.csv'

print(f"--- INICIANDO PROCESO ROBUSTO NBA ({TEMPORADA}) ---")

# 1. OBTENER CALENDARIO DE EQUIPOS (CORREGIDO)
print("Calculando partidos jugados por cada equipo (Base para cálculo de lesiones)...")

try:
    # Solicitamos clasificación
    standings = leaguestandings.LeagueStandings(season=TEMPORADA)
    df_standings = standings.get_data_frames()[0]
    
    # Aseguramos que el TeamID sea entero
    df_standings['TeamID'] = df_standings['TeamID'].astype(int)
    
    # CORRECCIÓN DE COLUMNAS: WINS y LOSSES
    total_partidos = df_standings['WINS'] + df_standings['LOSSES']
    
    # Diccionario { ID_Equipo : Total_Partidos }
    mapa_partidos_equipo = dict(zip(df_standings['TeamID'], total_partidos))
    
    # Validación
    if not mapa_partidos_equipo:
        raise ValueError("La API ha devuelto una tabla vacía.")
    
    print(f"Calendario cargado. Equipos procesados: {len(mapa_partidos_equipo)}")
    print(f"Validación de datos base (ID: Partidos): {list(mapa_partidos_equipo.items())[:3]}")

except Exception as e:
    print(f"[ERROR] CRÍTICO EN STANDINGS: {e}")
    print("Deteniendo ejecución para evitar datos corruptos.")
    raise e

# 2. OBTENER JUGADORES
todos_jugadores = players.get_active_players()

# 3. CHECKPOINT
ids_procesados = []
if os.path.exists(NOMBRE_ARCHIVO):
    try:
        df_existente = pd.read_csv(NOMBRE_ARCHIVO)
        # Verificamos columnas clave
        if 'Partidos_Lesionado' in df_existente.columns and 'PF' in df_existente.columns:
            ids_procesados = df_existente['PLAYER_ID'].unique().tolist()
            print(f"Retomando descarga. {len(ids_procesados)} jugadores ya procesados.")
        else:
            print("El archivo antiguo no es válido. Se reiniciará.")
    except:
        pass

jugadores_pendientes = [j for j in todos_jugadores if j['id'] not in ids_procesados]
print(f"Jugadores pendientes: {len(jugadores_pendientes)}")

# --- BUCLE PRINCIPAL ---
for i, jugador in enumerate(jugadores_pendientes):
    pid = jugador['id']
    nombre = jugador['full_name']
    
    print(f"[{i+1}/{len(jugadores_pendientes)}] {nombre}...", end=" ")
    
    try:
        # A. STATS
        gamelog = playergamelog.PlayerGameLog(player_id=pid, season=TEMPORADA, timeout=30)
        df_stats = gamelog.get_data_frames()[0]
        
        gp_jugador = 0
        stats_agrupadas = pd.DataFrame()
        columnas_stats = ['PTS', 'AST', 'REB', 'STL', 'BLK', 'TOV', 'PF', 'MIN', 'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA']

        if not df_stats.empty:
            cols_existentes = [c for c in columnas_stats if c in df_stats.columns]
            stats_agrupadas = df_stats[cols_existentes].sum().to_frame().T
            gp_jugador = len(df_stats)
        else:
            stats_agrupadas = pd.DataFrame(columns=columnas_stats)
            stats_agrupadas.loc[0] = 0
        
        stats_agrupadas['GP'] = gp_jugador
        stats_agrupadas['PLAYER_ID'] = pid
        
        time.sleep(0.6)

        # B. BIO
        bio_info = commonplayerinfo.CommonPlayerInfo(player_id=pid, timeout=30)
        df_bio = bio_info.get_data_frames()[0]
        
        cols_bio = ['PERSON_ID', 'DISPLAY_FIRST_LAST', 'COUNTRY', 'HEIGHT', 'POSITION', 'TEAM_ABBREVIATION', 'TEAM_ID', 'BIRTHDATE']
        cols_validas_bio = [c for c in cols_bio if c in df_bio.columns]
        df_bio_limpio = df_bio[cols_validas_bio].rename(columns={'PERSON_ID': 'PLAYER_ID'})
        
        # C. CÁLCULO DE AUSENCIAS
        if not df_bio_limpio.empty and 'TEAM_ID' in df_bio_limpio.columns:
            team_id = int(df_bio_limpio['TEAM_ID'].iloc[0])
            total_partidos_equipo = mapa_partidos_equipo.get(team_id, 0)
            
            if total_partidos_equipo > 0:
                ausencias = max(0, total_partidos_equipo - gp_jugador)
            else:
                ausencias = 0 
        else:
            ausencias = 0
            
        df_bio_limpio['Partidos_Lesionado'] = ausencias

        # D. MERGE
        df_final = pd.merge(df_bio_limpio, stats_agrupadas, on='PLAYER_ID', how='inner')
        
        if 'BIRTHDATE' in df_final.columns:
            try:
                df_final['Edad'] = 2026 - df_final['BIRTHDATE'].str[:4].astype(float)
            except:
                df_final['Edad'] = 0

        es_primero = not os.path.exists(NOMBRE_ARCHIVO)
        df_final.to_csv(NOMBRE_ARCHIVO, mode='a', header=es_primero, index=False)
        
        # LOG DE VALIDACIÓN
        print(f"[OK] (T: {total_partidos_equipo} - GP: {gp_jugador} = Aus: {ausencias})")
        
        if i % 10 == 0: time.sleep(2) 
        else: time.sleep(0.8)

    except Exception as e:
        print(f"[ERROR] {e}")
        time.sleep(5)

print(f"\nProceso finalizado. Archivo generado: {NOMBRE_ARCHIVO}")

--- INICIANDO PROCESO ROBUSTO NBA (2025-26) ---
Calculando partidos jugados por cada equipo (Base para cálculo de lesiones)...
Calendario cargado. Equipos procesados: 30
Validación de datos base (ID: Partidos): [(1610612760, 37), (1610612765, 36), (1610612738, 35)]
Retomando descarga. 464 jugadores ya procesados.
Jugadores pendientes: 66
[OK] (T: 37 - GP: 3 = Aus: 34)
[2/66] Karl-Anthony Towns... [OK] (T: 36 - GP: 33 = Aus: 3)
[3/66] Nolan Traore... [OK] (T: 33 - GP: 13 = Aus: 20)
[4/66] Luke Travers... [OK] (T: 38 - GP: 12 = Aus: 26)
[5/66] Gary Trent Jr.... [OK] (T: 36 - GP: 34 = Aus: 2)
[6/66] Oscar Tshiebwe... [OK] (T: 35 - GP: 1 = Aus: 34)
[7/66] Myles Turner... [OK] (T: 36 - GP: 36 = Aus: 0)
[8/66] Hunter Tyson... [OK] (T: 36 - GP: 13 = Aus: 23)
[9/66] Jaylon Tyson... [OK] (T: 38 - GP: 33 = Aus: 5)
[10/66] Jonas Valančiūnas... [OK] (T: 36 - GP: 33 = Aus: 3)
[11/66] Fred VanVleet... [OK] (T: 33 - GP: 0 = Aus: 33)
[12/66] Jarred Vanderbilt... [OK] (T: 34 - GP: 25 = Aus: 9)
[13/66] 